# 07 · Stacker with chunk-shape, seam and style features

The GhostWriter paper (Schaaf et al., LREC 2026, Section 3.2) builds a fusion text from a human text by masking up to
50% of its sentences or paragraphs and letting an LLM rewrite the gap. A fusion text is therefore mostly human with
one AI-written section, and two kinds of signal could show it:

* **Chunk shape.** The encoder of notebook 03 scores every 512-token chunk; notebook 04 used only the mean and the
  first chunk. If the AI section falls into one chunk, that chunk should look different from the rest: highest ai
  and fusion probability, lowest human probability, spread of the human probability across chunks.
* **Seams and style.** Where generated text was pasted in, formatting traces can remain. The EDA found one (a space
  before a closing quote, 8× more frequent in English fusion than in human texts); three similar traces and a few
  style statistics are added.

Reference: the stacker of notebook 04 (0.918 out-of-fold). Each variant adds one block of features; all scores are
out-of-fold on the same 5 group folds inside validation.

In [1]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-generated-text-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwkit import env
from hwkit.download import ensure_data
from hwkit.tracker import Run
P = env.setup(seed=42)
ensure_data(P)

platform: local
    data: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data
 outputs: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs
   cache: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs/cache
    repo: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection
  device: mps
data ready at /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data


PosixPath('/Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/data')

In [2]:
# Imports, config; validation and test texts; features of every block for both sets
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from hwkit.config import load_config
from hwkit.split import LABELS, load_split, scores
from hwkit.stack import chunk_features, group_folds, out_of_fold, text_features
from hwkit.submission import make_submission

NOTEBOOK_START = time.time()
cfg = load_config("stack")
METRICS = ["macro_f1_testlike", "macro_f1", "accuracy", "f1_human", "f1_fusion", "f1_ai"]

val = load_split(pd.read_csv(P.data / "train.csv", usecols=["id", "text", "label", "domain", "lang"]))
val = val[val["fold"] == 0].reset_index(drop=True)
test = pd.read_csv(P.data / "test.csv")
for df in (val, test):
    df["domain_lang"] = df["domain"] + " | " + df["lang"]
y = val["label"].map({c: i for i, c in enumerate(LABELS)}).to_numpy()
w = val["weight"].to_numpy()
folds = group_folds(val, cfg["cv"]["folds"])
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(val[["domain_lang"]])

def run_features(run_id, split, ids):
    """Chunk features of a saved encoder run, rows in the order of `ids`."""
    pred_dir = P.runs / run_id / "predictions"
    order = pd.read_csv(pred_dir / f"{split}.csv", usecols=["id"])["id"]
    f = chunk_features(dict(np.load(pred_dir / f"chunks_{split}.npz")), len(order), run_id)
    return f.set_index(order).loc[ids].reset_index(drop=True)

def blocks_for(split, df):
    tfidf = pd.read_csv(P.runs / cfg["inputs"]["baseline"] / "predictions" / f"{split}.csv").set_index("id")
    enc = pd.concat([run_features(r, split, df["id"]) for r in cfg["inputs"]["encoders"]], axis=1)
    base_cols = [c for c in enc.columns if "_mean_" in c or "_first_" in c or c.endswith("_log_chunks")]
    return {
        "base": pd.concat([enc[base_cols],
                           np.log(np.clip(tfidf.loc[df["id"], [f"s_{c}" for c in LABELS]], 1e-6, 1))
                           .reset_index(drop=True).add_prefix("tfidf_"),
                           pd.DataFrame(ohe.transform(df[["domain_lang"]]), columns=ohe.categories_[0])], axis=1),
        "shape": enc.drop(columns=base_cols),
        "text": text_features(df["text"]),
    }

t0 = time.time()
B = {"val": blocks_for("val", val), "test": blocks_for("test", test)}
run = Run("stack_features", cfg, P)
print({k: v.shape[1] for k, v in B["val"].items()}, f"features; {(time.time() - t0) / 60:.1f} min")

run 'stack_features' at /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/outputs/runs/stack_features  done: nothing yet
{'base': 19, 'shape': 5, 'text': 10} features; 0.3 min


In [3]:
# Does each new feature differ between human and fusion? Median per class on validation (unweighted)
B["val"]["shape"].join(B["val"]["text"]).assign(label=val["label"]).groupby("label").median().T[LABELS].round(3)

label,human,fusion,ai
encoder_xlmr_max_ai,0.001,0.000,1.000
encoder_xlmr_max_fusion,0.227,0.999,0.001
encoder_xlmr_min_human,0.762,0.000,0.000
encoder_xlmr_max_human,0.869,0.001,0.000
encoder_xlmr_std_human,0.006,0.000,0.000
seam_space_before_closing_quote,0.000,0.000,0.000
seam_space_before_punct,0.000,0.000,0.000
seam_double_space,0.000,0.000,0.000
seam_no_space_after_sentence,0.000,0.000,0.000
log_words,6.168,6.073,6.140


**Findings.** The chunk-shape features separate the classes clearly (lowest human probability across chunks: median
0.76 for human, 0.00 for fusion). The seam traces are 0 for the median text of every class, so they can only help in
the few texts that contain them. Fusion lies between human and ai in word length and vocabulary, as in the EDA.

## Variants, each against the reference

In [4]:
# Out-of-fold scores of each feature set and model; every variant is then refitted on all of validation for test
def logreg(scale=True):
    lr = LogisticRegression(C=cfg["logreg_C"], max_iter=5000)
    return make_pipeline(StandardScaler(), lr) if scale else lr

def boosting():
    return HistGradientBoostingClassifier(**cfg["boosting"], early_stopping=False, random_state=42)

def fit_model(make, X, rows):
    m = make()
    key = f"{m.steps[-1][0]}__sample_weight" if hasattr(m, "steps") else "sample_weight"
    return m.fit(X[rows], y[rows], **{key: w[rows]})

VARIANTS = {                                     # name: (feature blocks, model)
    "reference: notebook 04 stacker": (["base"], lambda: logreg(scale=False)),
    "same, standardized features": (["base"], logreg),
    "+ chunk shape": (["base", "shape"], logreg),
    "+ seams and style": (["base", "text"], logreg),
    "+ both": (["base", "shape", "text"], logreg),
    "reference features, gradient boosting": (["base"], boosting),
    "+ both, gradient boosting": (["base", "shape", "text"], boosting),
}
rows, oof, test_prob = [], {}, {}
for name, (blocks, make) in VARIANTS.items():
    t0 = time.time()
    Xv = pd.concat([B["val"][b] for b in blocks], axis=1).to_numpy()
    Xt = pd.concat([B["test"][b] for b in blocks], axis=1).to_numpy()
    oof[name] = out_of_fold(lambda r: fit_model(make, Xv, r), lambda m, r: m.predict_proba(Xv[r]), folds)
    test_prob[name] = fit_model(make, Xv, np.ones(len(val), dtype=bool)).predict_proba(Xt)
    res = scores(val["label"], np.array(LABELS)[oof[name].argmax(axis=1)], val["weight"])
    run.log_metrics(name, res)
    rows.append({"variant": name, **{k: res[k] for k in METRICS}, "minutes": (time.time() - t0) / 60})
summary = pd.DataFrame(rows)
summary.insert(1, "vs reference", summary["macro_f1_testlike"] - summary.loc[0, "macro_f1_testlike"])
summary.round(4)

,variant,vs reference,macro_f1_testlike,macro_f1,accuracy,f1_human,f1_fusion,f1_ai,minutes
0,reference: notebook 04 stacker,0.0000,0.9181,0.9201,0.9334,0.8736,0.8919,0.9948,0.1949
1,"same, standardized features",-0.0003,0.9179,0.9197,0.9331,0.8729,0.8916,0.9948,0.0267
2,+ chunk shape,0.0088,0.9270,0.9277,0.9397,0.8855,0.9031,0.9945,0.0314
3,+ seams and style,0.0008,0.9190,0.9207,0.9338,0.8751,0.8925,0.9946,0.0339
4,+ both,0.0088,0.9269,0.9276,0.9397,0.8853,0.9030,0.9947,0.0365
5,"reference features, gradient boosting",0.0142,0.9323,0.9345,0.9450,0.8984,0.9115,0.9937,2.6336
6,"+ both, gradient boosting",0.0222,0.9404,0.9428,0.9520,0.9113,0.9219,0.9951,2.7585


**Findings.** Standardizing changes nothing. Chunk shape adds +0.009 to the linear stacker, seams and style only
+0.001. Gradient boosting on the reference features alone adds +0.014, so the step from linear to non-linear
combination matters; the new features add another +0.008 on top (0.940 in total, +0.022 over notebook 04).

In [5]:
# Macro-F1 per domain (over the classes present there): reference vs the best variant
best = summary.iloc[1:].sort_values("macro_f1_testlike").iloc[-1]["variant"]
dom = pd.DataFrame({name: {dl: f1_score(g["label"], np.array(LABELS)[oof[name][g.index].argmax(axis=1)],
                                        labels=sorted(g["label"].unique()), average="macro", zero_division=0)
                           for dl, g in val.groupby("domain_lang")}
                    for name in (summary.loc[0, "variant"], best)})
dom.columns = ["reference", "best"]
dom["texts"] = val["domain_lang"].value_counts()
dom["change"] = dom["best"] - dom["reference"]
print("best variant:", best)
dom.round(3).sort_values("change")

best variant: + both, gradient boosting


,reference,best,texts,change
student_essays | en-EN,0.934,0.937,5411,0.003
blog_authorship_corpus | en-EN,0.861,0.865,4053,0.005
cnn_news | en-EN,0.967,0.984,4429,0.017
bundestag | de-DE,0.924,0.957,5234,0.033
house_of_commons | en-EN,0.921,0.954,3186,0.033
euro_court_cases | en-EN,0.908,0.962,2759,0.053
gutenberg | en-EN,0.773,0.912,692,0.139
arxiv_papers | en-EN,0.740,0.938,1429,0.198
gutenberg | de-DE,0.685,0.905,43,0.220


**Findings.** The gain is largest in arxiv (+0.20) and gutenberg (+0.14 en, +0.22 de with only 43 texts), the
domains with a strong length difference between classes (EDA). Gradient boosting can learn domain-specific rules
such as "a long arxiv text is not ai", which a linear model cannot. Blogs and student essays barely change.

In [6]:
# Save out-of-fold and test probabilities of the best variant; write its submission file
cols = [f"s_{c}" for c in LABELS]
pd.concat([val[["id", "label"]], pd.DataFrame(oof[best], columns=cols)], axis=1).to_csv(run.pred_path("val_oof"), index=False)
pd.concat([test[["id"]], pd.DataFrame(test_prob[best], columns=cols)], axis=1).to_csv(run.pred_path("test"), index=False)
sub = make_submission(test["id"], np.array(LABELS)[test_prob[best].argmax(axis=1)],
                      P.repo / "submissions" / "stack_features.csv", P.data)

submission saved: /Users/anna/Desktop/AI/ucu-ai-hw-generated-text-detection/submissions/stack_features.csv  (25000 rows)
label
ai        0.513
human     0.267
fusion    0.220


In [7]:
# Distinct warnings logged in this session (should be empty), total time
print(f"total: {(time.time() - NOTEBOOK_START) / 60:.1f} min")
env.warnings_summary(P)

total: 8.1 min


[]